In [6]:
import sys
import pathlib
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent  # points to repo root
sys.path.insert(0, str(project_root))

from app.environment.tennis_env import TennisEnv
from app.agents.dqn_agent import DQNAgent
from app.training.dqn_trainer import DQNTrainer as Trainer
from app.data.transition_graph import TransitionBuilder


"""Main training script for tennis RL agent"""

# Set up paths
data_path = project_root / "data" / "processed" / "shot_transitions_combined.csv"

print("Building transition graph...")
# Build transition graph
graph_builder = TransitionBuilder(
    transitions_path=str(data_path),
    temperature=1.0
)
transition_graph = graph_builder.build()
print("Transition graph built successfully!")

# Create environment
print("Creating tennis environment...")
env = TennisEnv(
    transition_graph=transition_graph,
    serve_first=True,
    
)

# Create DQN agent
print("Initializing DQN agent...")
agent = DQNAgent(
    env=env,
    lr=0.001,
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.01,
    epsilon_decay=0.999,
    memory_size=10000,
    batch_size=32,
    target_update_freq=100
)

# Create trainer
trainer = Trainer(
    env=env,
    agent=agent,
    mlflow_tracking_uri="https://mlflow.digi.com.br",
    experiment_name="tennis-rl-dqn"
)

# Training configuration
training_config = {
    "episodes": 100,
    "save_freq": 100,
    "eval_freq": 200,
    "run_name": "dqn_tennis_v1",
    "tags": {
        "model_type": "DQN",
        "environment": "tennis",
        "data_source": "charting-m-points-2020s",
        "temperature": "1.0"
    }
}

print(f"Starting training for {training_config['episodes']} episodes...")

Building transition graph...
Transition graph built successfully!
Creating tennis environment...
Initializing DQN agent...
Starting training for 100 episodes...


In [7]:
print(agent.state_size, agent.action_size)

29 54


In [33]:
import random
agent.memory.clear()

for i in range(300):
    action = agent.act(env.state)
    next_state, reward, done, info = env.step(action)
    agent.remember(env.state, random.randint(1,17), reward, next_state, done)
    env.state = next_state
    if done:
        env.reset()
        break

Ação ilegal detectada: shot_type='r' shot_direction=3
Ação ilegal detectada: shot_type='t' shot_direction=2
Ação ilegal detectada: shot_type='h' shot_direction=2
Ação ilegal detectada: shot_type='j' shot_direction=3
Ação ilegal detectada: shot_type='p' shot_direction=2
Ação ilegal detectada: shot_type='k' shot_direction=1
Ação ilegal detectada: shot_type='h' shot_direction=3
Ação ilegal detectada: shot_type='k' shot_direction=2
Ação ilegal detectada: shot_type='p' shot_direction=2
Ação ilegal detectada: shot_type='k' shot_direction=1
Ação ilegal detectada: shot_type='j' shot_direction=1
Ação ilegal detectada: shot_type='y' shot_direction=2
Ação ilegal detectada: shot_type='y' shot_direction=3
Ação ilegal detectada: shot_type='i' shot_direction=1
Ação ilegal detectada: shot_type='m' shot_direction=2
Ação ilegal detectada: shot_type='h' shot_direction=1
Ação ilegal detectada: shot_type='l' shot_direction=3
Ação ilegal detectada: shot_type='i' shot_direction=2
Ação ilegal detectada: shot_

In [9]:
env.state

State(last_shot_type='winner', last_shot_direction=2, player_game_score='15', player_set_score=3, pc_game_score='0', pc_set_score=1, player_serves=True)

In [10]:
print(env.step(action))

Ação ilegal detectada: shot_type='i' shot_direction=2
(State(last_shot_type='winner', last_shot_direction=2, player_game_score='15', player_set_score=3, pc_game_score='0', pc_set_score=1, player_serves=True), -5, False, {})


In [ ]:
env.state.encode(env)  # Test state encoding

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 3,
 1,
 3,
 0,
 1]

In [11]:
import torchinfo

torchinfo.summary(agent.q_network, input_size=(1, agent.state_size))

Layer (type:depth-idx)                   Output Shape              Param #
DQNNetwork                               [1, 54]                   --
├─Linear: 1-1                            [1, 512]                  15,360
├─Linear: 1-2                            [1, 512]                  262,656
├─Linear: 1-3                            [1, 512]                  262,656
├─Linear: 1-4                            [1, 54]                   27,702
Total params: 568,374
Trainable params: 568,374
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.57
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 2.27
Estimated Total Size (MB): 2.29

In [31]:
agent.q_network

DQNNetwork(
  (fc1): Linear(in_features=29, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=512, bias=True)
  (fc4): Linear(in_features=512, out_features=54, bias=True)
)

In [16]:
import random
import torch
import torch.nn as nn

In [34]:
batch = random.sample(agent.memory, agent.batch_size)
states = torch.FloatTensor([e[0].encode(agent.env) for e in batch]).to(
    agent.device
)
actions = torch.LongTensor([e[1] for e in batch]).to(agent.device)
rewards = torch.FloatTensor([e[2] for e in batch]).to(agent.device)
next_states = torch.FloatTensor([e[3].encode(agent.env) for e in batch]).to(
    agent.device
)
dones = torch.BoolTensor([e[4] for e in batch]).to(agent.device)

# Q Values = Q(s, a)
current_q_values = agent.q_network(states).gather(1, actions.unsqueeze(1))

# Q Values next = max_a' Q_target(s', a')
next_q_values = agent.target_network(next_states).max(1)[0].detach()

# Target = reward + gamma * max_a' Q_target(s', a') * (1 - done)
target_q_values = rewards + (agent.gamma * next_q_values * ~dones)

loss = nn.MSELoss()(current_q_values.squeeze(), target_q_values)

agent.optimizer.zero_grad()
loss.backward()
agent.optimizer.step()

# Update target network
agent.step_count += 1
if agent.step_count % agent.target_update_freq == 0:
    agent.update_target_network()

# Decay epsilon
if agent.epsilon > agent.epsilon_min:
    agent.epsilon *= agent.epsilon_decay

loss.item()

19.481426239013672

In [43]:
agent.q_network(states).gather(1, actions.unsqueeze(1))

tensor([[-0.0412],
        [-0.0083],
        [-0.0624],
        [-0.0960],
        [ 0.0085],
        [-0.0412],
        [-0.0801],
        [ 0.0085],
        [ 0.0068],
        [-0.0635],
        [-0.0179],
        [-0.0672],
        [-0.0495],
        [-0.0745],
        [ 0.0085],
        [-0.0635],
        [-0.0624],
        [-0.0960],
        [-0.0624],
        [-0.0083],
        [ 0.0121],
        [-0.0635],
        [-0.0083],
        [ 0.0121],
        [-0.0206],
        [-0.1057],
        [-0.0206],
        [-0.0389],
        [-0.0083],
        [-0.0495],
        [-0.0672],
        [-0.0960]], device='cuda:0', grad_fn=<GatherBackward0>)

In [40]:
next_q_values

tensor([0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522,
        0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522,
        0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522, 0.0522,
        0.0522, 0.0522, 0.0522, 0.0522, 0.0522], device='cuda:0')